<a href="https://colab.research.google.com/github/benzsevern/benzsevern/blob/main/MOFA_Video_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MOFA-Video -> animate the montage photos (Colab)

Animates each still in `The Story of Us` into a short motion clip with **MOFA-Video** (trajectory mode), driven by *synthetic* motion hints so you don't hand-draw anything. The clips land in your Google Drive; the local `super8_montage.py` then scrolls them through the filmstrip.

## Read this first
- **Runtime > Change runtime type > GPU.** This is Stable-Video-Diffusion under the hood. A **free T4 (16GB) is borderline and may OOM**; an **L4 / A100 (Colab Pro) is strongly recommended**. Expect roughly 1-4 min per photo, so ~44 photos = a while.
- It's **resumable**: clips already in the output folder are skipped, so a disconnect won't make you start over. Just re-run the last cell.
- First run downloads ~10GB of weights.
- This notebook bypasses MOFA's Gradio UI by importing its `Drag` class directly and feeding generated trajectories.

In [14]:
!nvidia-smi

Sat May 30 20:52:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Mount Drive and set your paths
Upload the `The Story of Us` photo folder to your Drive first, then edit `INPUT_DIR` below if needed.

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os

# --- EDIT THESE if your Drive layout differs ---
INPUT_DIR  = '/content/drive/MyDrive/The Story of Us'            # where your photos live
OUTPUT_DIR = '/content/drive/MyDrive/The Story of Us/mofa_clips' # animated clips go here

# --- motion / model settings ---
WIDTH, HEIGHT = 384, 512   # portrait, multiples of 64 (matches the filmstrip cells)
MODEL_LENGTH  = 25         # frames MOFA generates per photo (~1.25s @ 20fps)
CTRL_SCALE    = 0.6        # how strongly the motion hint is followed (0.4-0.8 is sane)
MOTION        = 'zoom'     # 'zoom' (slow push-in), 'sway' (horizontal), or 'drift' (down)

os.makedirs(OUTPUT_DIR, exist_ok=True)
assert os.path.isdir(INPUT_DIR), f'INPUT_DIR not found: {INPUT_DIR}'
print('photos folder OK, clips ->', OUTPUT_DIR)

photos folder OK, clips -> /content/drive/MyDrive/The Story of Us/mofa_clips


In [17]:
import getpass
from huggingface_hub import login, whoami, snapshot_download
tok = getpass.getpass("Paste HF READ token (input hidden): ")
login(token=tok)
print("logged in as:", whoami()["name"])
WORK = '/content/MOFA-Video/MOFA-Video-Traj'
snapshot_download('stabilityai/stable-video-diffusion-img2vid-xt-1-1',
                  local_dir=f'{WORK}/ckpts/stable-video-diffusion-img2vid-xt-1-1')
print("SVD downloaded OK")

Paste HF READ token (input hidden): ··········
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful
logged in as: benzsevern


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

SVD downloaded OK


## 2. Clone MOFA-Video and install deps

In [18]:
%cd /content
!git lfs install
![ -d MOFA-Video ] || git clone https://github.com/MyNiuuu/MOFA-Video.git
WORK = '/content/MOFA-Video/MOFA-Video-Traj'
%cd $WORK
!pip -q install -r requirements.txt
!pip -q install opencv-python-headless huggingface_hub

/content
Git LFS initialized.
/content/MOFA-Video/MOFA-Video-Traj
ERROR: Could not find a version that satisfies the requirement torch==2.0.1 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0)
ERROR: No matching distribution found for torch==2.0.1


## 3. Download checkpoints (~10GB, first run only)
Pulls the MOFA-Traj adapter + CMP from HuggingFace and SVD-xt from Stability, then merges them into the working tree where the code expects them.

In [19]:
import glob, shutil
from huggingface_hub import snapshot_download

WORK = '/content/MOFA-Video/MOFA-Video-Traj'

# MOFA-Traj HF repo mirrors ckpts/ (adapter+controlnet) and models/cmp/...
snapshot_download('MyNiuuu/MOFA-Video-Traj', local_dir='/content/mofa_hf')
for sub in ('ckpts', 'models'):
    s = f'/content/mofa_hf/{sub}'
    if os.path.isdir(s):
        shutil.copytree(s, f'{WORK}/{sub}', dirs_exist_ok=True)

# Stable Video Diffusion xt 1.1 -> ckpts/
snapshot_download('stabilityai/stable-video-diffusion-img2vid-xt-1-1',
                  local_dir=f'{WORK}/ckpts/stable-video-diffusion-img2vid-xt-1-1')

# sanity: the code wants ckpts/controlnet and the CMP .pth.tar
print('--- ckpts/ ---')
for p in sorted(glob.glob(f'{WORK}/ckpts/*')): print(' ', p)
cmp = glob.glob(f'{WORK}/**/ckpt_iter_42000.pth.tar', recursive=True)
print('CMP checkpoint:', cmp or 'NOT FOUND -- check models/cmp path')
print('controlnet:', glob.glob(f'{WORK}/ckpts/controlnet') or 'NOT FOUND')

Fetching 105 files:   0%|          | 0/105 [00:00<?, ?it/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

--- ckpts/ ---
  /content/MOFA-Video/MOFA-Video-Traj/ckpts/controlnet
  /content/MOFA-Video/MOFA-Video-Traj/ckpts/stable-video-diffusion-img2vid-xt-1-1
CMP checkpoint: ['/content/MOFA-Video/MOFA-Video-Traj/models/cmp/experiments/semiauto_annot/resnet50_vip+mpii_liteflow/checkpoints/ckpt_iter_42000.pth.tar']
controlnet: ['/content/MOFA-Video/MOFA-Video-Traj/ckpts/controlnet']


## 4. Make MOFA importable without launching the Gradio server
Strips everything from the first Gradio UI line onward into `mofa_core.py`, so `from mofa_core import Drag` won't start a web server.

In [20]:
WORK = '/content/MOFA-Video/MOFA-Video-Traj'
src = open(f'{WORK}/run_gradio.py', 'r', encoding='utf-8').read()
cut = len(src)
for marker in ('\nwith gr.Blocks', '\ndemo = gr.Blocks', '\ndemo=gr.Blocks', "\nif __name__"):
    i = src.find(marker)
    if i != -1:
        cut = min(cut, i)
open(f'{WORK}/mofa_core.py', 'w', encoding='utf-8').write(src[:cut])
print(f'wrote mofa_core.py ({cut} of {len(src)} chars kept)')

wrote mofa_core.py (26555 of 37794 chars kept)


In [21]:
!pip -q install colorlog einops av omegaconf

In [22]:
import os, glob
print("INPUT_DIR =", INPUT_DIR, "exists:", os.path.isdir(INPUT_DIR))
print("contents:", os.listdir(INPUT_DIR)[:10] if os.path.isdir(INPUT_DIR) else "—")
# search Drive for the folder if the path is off:
print(glob.glob('/content/drive/MyDrive/**/The Story of Us', recursive=True))

INPUT_DIR = /content/drive/MyDrive/The Story of Us exists: True
contents: ['mofa_clips', 'Facetune_29-12-2021-17-41-00.JPEG', 'IMG_4936.JPEG', 'IMG_3925.JPEG', 'IMG_4337.JPEG', 'IMG_5003.JPEG', 'IMG_4821.JPEG', 'IMG_5278.JPEG', 'IMG_2123.JPEG', 'IMG_7416.JPEG']
['/content/drive/MyDrive/The Story of Us']


In [23]:
import diffusers, huggingface_hub
print("diffusers", diffusers.__version__)
print("hub", huggingface_hub.__version__)

print("diffusers", diffusers.__version__)
print("hub", huggingface_hub.__version__)

diffusers 0.24.0
hub 0.23.4
diffusers 0.24.0
hub 0.23.4


In [24]:
import diffusers, transformers, accelerate, huggingface_hub
print(diffusers.__version__, transformers.__version__, accelerate.__version__, huggingface_hub.__version__)

ImportError: tokenizers>=0.19,<0.20 is required for a normal functioning of this module, but found tokenizers==0.22.2.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

## 5. Animate every photo
Synthetic trajectories give MOFA a gentle motion hint; SVD fills in plausible movement (hair, water, parallax). Resumable: re-run if you get disconnected.

In [ ]:
%cd /content/MOFA-Video/MOFA-Video-Traj
import os, shutil, numpy as np, torch
from PIL import Image, ImageOps
from mofa_core import Drag

class FakeState:
    """run_gradio reads tracking_points.constructor_args['value']."""
    def __init__(self, value):
        self.constructor_args = {'value': value}

def make_trajectories(W, H, mode):
    """A 3x3 grid of points, each with a start->end implying gentle motion."""
    cx, cy = W / 2, H / 2
    seeds = [(x, y) for x in (W * 0.25, W * 0.5, W * 0.75)
                    for y in (H * 0.25, H * 0.5, H * 0.75)]
    traj = []
    for (x, y) in seeds:
        if mode == 'zoom':      # slow push-in: drift outward from center
            ex, ey = x + (x - cx) * 0.14, y + (y - cy) * 0.14
        elif mode == 'sway':    # gentle horizontal sway
            ex, ey = x + W * 0.04, y
        else:                   # 'drift': slight downward parallax
            ex, ey = x, y + H * 0.05
        traj.append([(int(x), int(y)), (int(ex), int(ey))])
    return traj

IMG_EXTS = ('.jpg', '.jpeg', '.png')
photos = sorted(f for f in os.listdir(INPUT_DIR) if f.lower().endswith(IMG_EXTS))
print(len(photos), 'photos found')

drag = Drag('cuda', HEIGHT, WIDTH, MODEL_LENGTH)
mask = np.zeros((HEIGHT, WIDTH), np.uint8)        # empty motion-brush -> global motion
viz  = np.zeros((HEIGHT, WIDTH, 3), np.uint8)
os.makedirs('/content/mofa_in', exist_ok=True)

for idx, name in enumerate(photos):
    out_clip = os.path.join(OUTPUT_DIR, f'{idx:04d}.mp4')
    if os.path.exists(out_clip):
        print('skip (already done):', name); continue
    inp = f'/content/mofa_in/{idx:04d}_frame.png'   # id parsed as '{idx:04d}'
    im = ImageOps.exif_transpose(Image.open(os.path.join(INPUT_DIR, name))).convert('RGB')
    ImageOps.fit(im, (WIDTH, HEIGHT), Image.LANCZOS).save(inp)
    tps = FakeState(make_trajectories(WIDTH, HEIGHT, MOTION))
    try:
        ret = drag.run(inp, tps, 1, mask, viz, CTRL_SCALE)
        shutil.copyfile(ret[3], out_clip)   # ret[3] = outputs_mp4_path
        print(f'[{idx + 1}/{len(photos)}] {name} -> {out_clip}')
    except Exception as e:
        print('FAILED', name, '::', repr(e))
    torch.cuda.empty_cache()

print('Done. Animated clips in', OUTPUT_DIR)

## 6. Done
The clips are named `0000.mp4 ... 00NN.mp4` in `OUTPUT_DIR` (same index order as the sorted photos). Download that `mofa_clips` folder from Drive, then run locally:

```
python super8_montage.py "C:\\Users\\bsevern\\Downloads\\The Story of Us" --clips "C:\\path\\to\\mofa_clips"
```

(the `--clips` mode gets wired up on the local side next).